# Phase 3 retained-pilot diagnostics

**Evidence status:** `pilot_adaptation` / `not_assessed`. This notebook reads retained ledgers and result records only. It neither runs a decoder nor makes a reproduction, ranking, superiority, or equal-total-cost claim.

Work counters have different units and are displayed separately; no scalar total work score is defined.

In [ ]:
import csv
import hashlib
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(os.environ['PHASE3_REPOSITORY_ROOT']).resolve()
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'bin' / 'python').resolve()
EVIDENCE_ROOT = ROOT / 'data' / 'results' / 'phase3-pilots'
REPORT_ROOT = ROOT / 'reports' / 'phase3-pilots'
RESULT_PATH = EVIDENCE_ROOT / 'result.json'
EVENTS_PATH = EVIDENCE_ROOT / 'events.jsonl'
for required in (RESULT_PATH, EVENTS_PATH):
    assert required.is_file(), f'missing retained evidence: {required}'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
LABEL = 'pilot_adaptation / not_assessed — diagnostic only; no reproduction, ranking, or equal-total-cost claim'


In [ ]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
assert result['relation'] == 'pilot_adaptation'
assert result['assessment_status'] == 'not_assessed'
assert 'not statistical curve reproductions' in result['statistical_claim']
terminals = result['verified_terminals']
events = [json.loads(line) for line in EVENTS_PATH.read_text(encoding='utf-8').splitlines() if line]
assert terminals and events
assert {event['kind'] for event in events} >= {'shot_begin', 'work', 'runtime', 'termination'}
assert all(item['relation'] == 'pilot_adaptation' and item['assessment_status'] == 'not_assessed' for item in terminals)
assert all(len(item['artifact_raw_sha256']) == 64 and len(item['artifact_semantic_sha256']) == 64 and len(item['composition_identity']) == 64 for item in terminals)
terminal_events = {(event['run_id'], event['shot_id']): event for event in events if event['kind'] == 'termination'}
sample_events = {(event['run_id'], event['shot_id']): event for event in events if event['kind'] == 'shot_begin'}
assert len(terminal_events) == len(sample_events) == len(terminals)
for item in terminals:
    key = (item['run_id'], item['shot_id'])
    event = terminal_events[key]
    data = event['data']
    assert item['termination'] == event['termination']
    assert item['sample_identity'] == sample_events[key]['sample_seed_identity']
    for field in ('code_id', 'artifact_raw_sha256', 'artifact_semantic_sha256', 'canonical_identity', 'composition_identity', 'outcome', 'converged', 'work_vector', 'policy_effects', 'setup_runtime', 'decode_runtime'):
        assert item[field] == data[field], f'retained terminal differs from ledger: {field}'
evidence_digest = hashlib.sha256(RESULT_PATH.read_bytes() + EVENTS_PATH.read_bytes()).hexdigest()
assert all(item['budget_views']['equal_total_cost_claim'] is False for item in terminals)
len(terminals), Counter(item['termination'] for item in terminals)


In [ ]:
def policy_binding(item, role):
    return item['policy_effects'][role]['binding_id']

def trace_link(item):
    return f'data/results/phase3-pilots/events.jsonl#run_id={item["run_id"]};shot_id={item["shot_id"]}'

outcome_columns = [
    'run_id', 'code_id', 'update_policy', 'schedule_policy', 'termination', 'converged',
    'outcome', 'outcome_missing_reason', 'qualified_iterations', 'common_c2v_cap',
    'schedule_qualified_iteration_limit', 'equal_total_cost_claim', 'setup_runtime_ns',
    'decode_runtime_ns', 'denominator_shots', 'cohort_terminal_count', 'trace_evidence_link', 'evidence_sha256', 'relation', 'assessment_status',
]
work_columns = [
    'run_id', 'code_id', 'update_policy', 'schedule_policy', 'qualified_iterations',
    'full_work_vector_json', 'check_to_variable_updates',
    'variable_to_check_updates', 'convergence_checks', 'residual_priority_evaluations',
    'setup_runtime_ns', 'decode_runtime_ns', 'common_c2v_cap',
    'schedule_qualified_iteration_limit', 'equal_total_cost_claim', 'trace_evidence_link',
    'evidence_sha256', 'relation', 'assessment_status',
]
outcome_rows, work_rows = [], []
for item in terminals:
    work = item['work_vector']
    outcome = item['outcome']
    base = {
        'run_id': item['run_id'], 'code_id': item['code_id'],
        'update_policy': policy_binding(item, 'update'),
        'schedule_policy': policy_binding(item, 'schedule'),
        'qualified_iterations': json.dumps(item['qualified_iterations'], sort_keys=True),
        'common_c2v_cap': item['budget_views']['common_c2v_cap'],
        'schedule_qualified_iteration_limit': item['budget_views']['qualified_iteration_limit'],
        'equal_total_cost_claim': item['budget_views']['equal_total_cost_claim'],
        'setup_runtime_ns': item['setup_runtime']['value'],
        'decode_runtime_ns': item['decode_runtime']['value'],
        'trace_evidence_link': trace_link(item), 'evidence_sha256': evidence_digest,
        'relation': item['relation'], 'assessment_status': item['assessment_status'],
    }
    outcome_rows.append({**base, 'termination': item['termination'], 'converged': item['converged'],
        'outcome': outcome['class'] if outcome else 'missing',
        'outcome_missing_reason': 'unavailable_due_to_stopping' if outcome is None else '',
        'denominator_shots': sum(other['run_id'] == item['run_id'] for other in terminals),
        'cohort_terminal_count': len(terminals)})
    work_rows.append({**base, 'full_work_vector_json': json.dumps(work, sort_keys=True),
        'check_to_variable_updates': work.get('check_to_variable_updates', 0),
        'variable_to_check_updates': work.get('variable_to_check_updates', 0),
        'convergence_checks': work.get('convergence_checks', 0),
        'residual_priority_evaluations': work.get('residual_priority_evaluations', 0)})

for path, columns, rows in ((REPORT_ROOT / 'outcomes.csv', outcome_columns, outcome_rows), (REPORT_ROOT / 'work-by-policy.csv', work_columns, work_rows)):
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)
assert all((REPORT_ROOT / name).stat().st_size > 0 for name in ('outcomes.csv', 'work-by-policy.csv'))


In [ ]:
labels = [item['run_id'].replace('phase3-', '') for item in terminals]
converged = [int(item['converged']) for item in terminals]
fig, axis = plt.subplots(figsize=(12, 4))
axis.bar(range(len(labels)), converged, color='#4477aa')
axis.set_xticks(range(len(labels)), labels, rotation=70, ha='right', fontsize=7)
axis.set_ylim(0, 1.1)
axis.set_yticks([0, 1])
if not any(converged):
    axis.text(0.5, 0.55, f'0/{len(converged)} pilots converged within the declared one-iteration limit', transform=axis.transAxes, ha='center', va='center', fontsize=11)
axis.set_ylabel('converged (retained pilot observation)')
axis.set_title('Phase 3 pilot adaptation — convergence (not assessed)')
fig.tight_layout()
fig.savefig(REPORT_ROOT / 'convergence.png', dpi=160, metadata={'Title': 'Phase 3 ' + LABEL, 'Description': 'Retained evidence diagnostic; not a ranking or reproduction assessment.'})
plt.close(fig)

work_series = (
    ('check_to_variable_updates', 'C2V updates'),
    ('variable_to_check_updates', 'V2C updates'),
    ('convergence_checks', 'convergence checks'),
    ('residual_priority_evaluations', 'residual priority evaluations'),
)
setup_ms = [row['setup_runtime_ns'] / 1_000_000 for row in work_rows]
decode_ms = [row['decode_runtime_ns'] / 1_000_000 for row in work_rows]
fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
for axis, (field, title) in zip(axes[:4], work_series):
    values = [row[field] for row in work_rows]
    axis.bar(range(len(labels)), values, color='#66aa55')
    axis.set_ylim(0, max(1, max(values) * 1.1))
    axis.set_ylabel(title)
    axis.grid(axis='y', alpha=0.2)
axes[0].set_title('Phase 3 pilot adaptation — separate work and runtime (not assessed)')
axes[4].bar([index - 0.18 for index in range(len(labels))], setup_ms, width=0.36, label='setup', color='#cc6677')
axes[4].bar([index + 0.18 for index in range(len(labels))], decode_ms, width=0.36, label='decode', color='#ddcc77')
axes[4].set_ylabel('runtime (ms)')
axes[4].set_xticks(range(len(labels)), labels, rotation=70, ha='right', fontsize=7)
axes[4].legend()
fig.tight_layout()
fig.savefig(REPORT_ROOT / 'work-and-runtime.png', dpi=160, metadata={'Title': 'Phase 3 ' + LABEL, 'Description': 'Different work counters have different units and are never summed; setup/decode timings are separate.'})
plt.close(fig)
assert all((REPORT_ROOT / name).stat().st_size > 0 for name in ('convergence.png', 'work-and-runtime.png'))


## Interpretation boundary

The tables and plots preserve missing outcomes, schedule-qualified iterations, full atomic work, and separate setup/decode timing scopes. They are bounded `pilot_adaptation` evidence with `not_assessed` status. Equal message caps or iteration limits do **not** imply equal total cost; no result here is a reproduction assessment or a decoder ranking.